In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("./dfs/final_df.csv")
df.sample(3)

,question,evidence,ground_truth,url,source
12,How does the arborescence-based training model...,"[""Graph-based Dissimilarity Let Gbe a graph wi...",The model calculates dissimilarity using a min...,https://aclanthology.org/2022.naacl-main.343.pdf,1
15,"In the Oracle Inference experiment, how do the...","[""Finally, we also provide representative exam...","The ""SELF"" setting isolates re-ranking capabil...",https://aclanthology.org/2022.naacl-main.343.pdf,1
2,How does GraphVLM relate to the activity graph...,"['Lastly , we introduce GraphVLM as our framew...",GraphVLM is the model we developed that predic...,https://openreview.net/forum?id=eJhc_CPXQIT,3


## Print Query Hierarchy Chunks Retrieval

In [11]:
import importlib
import main

importlib.reload(main)
run_hierarchy_pipeline = main.run_hierarchy_pipeline

test_query = "What is the difference between the MOMA dataset and the MOMA-LRG dataset?"
answer, hierarchy_template, retrieved_content, decision = run_hierarchy_pipeline(query=test_query, vector_db_path="./outputs/test3/", top_k=1)

Tree PNG saved -> decompositions\test6\decomposition_What_is_the_differen_240a29ff-1cdd-431a-9995-f45fb15e8ac2.png


In [13]:
decision

RoutingDecision(route='MULTI_HOP', confidence=0.65, reason='Need to compare two datasets and identify their differences.')

In [12]:
answer

'The MOMA-LRG dataset differs from the original MOMA dataset in several key ways across data, tasks, and labeling approach:\n\n- Scale and scope of data/tasks:\n  - MOMA-LRG is a substantially larger dataset with an order of magnitude more annotations and longer videos from a wider variety of scenes, designed to support a more comprehensive representation of human activity.\n  - It introduces activity graphs that span three levels of hierarchy (not just atomic actions), providing a unified graphical framework for understanding complex activities.\n  - By contrast, MOMA focuses on atomic actions and does not adopt this multi-level, graph-based overarching representation.\n\n- Grounding, entity disambiguation, and role labeling:\n  - MOMA-LRG grounds all associated entities and disambiguates which entities are involved in each interaction when multiple entities are present. It also explicitly classifies each actor’s role, recognizing that different roles can be informative for interpreti

In [15]:
print(hierarchy_template)

1. What aspects differentiate MOMA and MOMA-LRG datasets across data and tasks?
1.1. What data sources are used by MOMA vs MOMA-LRG?
1.1.1. What data sources are used by MOMA?
        retrieved content:
          retrieved chunk: Comparison with existing datasets. Compared to existing datasets, there are several key advantages that the MOMA-LRG dataset provides. First, MOMA-LRG grounds all associated entities. In contexts with more than one entity, it is necessary to disambiguate which entities are involved in a particular interaction. Existing ego-centric datasets [16, 27] dodge this issue since at most interaction is involved in a scene. Second, we classify each actor’s role. Typical datasets [27, 28] involve one type of actor and hence do not label the person’s role [25, 19, 21]. In a diverse set of scenes, the role of the actor becomes more important in understanding actions since it can provide an important signal in parsing a human activity [91]. Third, the MOMA-LRG dataset diffe

In [14]:
retrieved_content

['Comparison with existing datasets. Compared to existing datasets, there are several key advantages that the MOMA-LRG dataset provides. First, MOMA-LRG grounds all associated entities. In contexts with more than one entity, it is necessary to disambiguate which entities are involved in a particular interaction. Existing ego-centric datasets [16, 27] dodge this issue since at most interaction is involved in a scene. Second, we classify each actor’s role. Typical datasets [27, 28] involve one type of actor and hence do not label the person’s role [25, 19, 21]. In a diverse set of scenes, the role of the actor becomes more important in understanding actions since it can provide an important signal in parsing a human activity [91]. Third, the MOMA-LRG dataset differentiates between static and dynamic predicates. For example, the dynamic predicate sitting down is a dynamic movement where an actor transitions from the standing static predicate to the sitting static predicate. We argue that 

## Querying

In [17]:
from datetime import datetime

db_dir = {
    "1": "./outputs/test1/",
    "2": "./outputs/test2/",
    "3": "./outputs/test3/"
}

In [18]:
start = datetime.now()
all_answers = []
qa_latencies = []

for db_id, input_dir in db_dir.items():
    print(f"Running pipeline for DB {db_id}...")
    rows = df[df["source"] == int(db_id)]

    for _, row in rows.iterrows():
        question = row["question"]
        print(f"Question: {question}")
        s = datetime.now()
        answer, hierarchy_template, retrieved_content, decision = run_hierarchy_pipeline(query=question, vector_db_path=input_dir, top_k=3)
        print(f"Time taken for question: {(datetime.now() - s).seconds} seconds")
        qa_latencies.append((datetime.now() - s).seconds)
        unique_docs = set(retrieved_content)
        all_answers.append({
                "question": question,
                "answer": answer,
                "context": unique_docs,
                "ground_truth": row["ground_truth"],
                "evidence": row["evidence"],
                "source": db_id
            })
end = datetime.now()
ellapsed_time = end - start
print(f"Total time taken: {ellapsed_time}")

Running pipeline for DB 1...
Question: What is the target KB size used for MedMentions in the experiments?
Time taken for question: 9 seconds
Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Tree PNG saved -> decompositions\test6\decomposition_Do_the_training_and__67784154-7d23-44a2-9447-1e7c845471e4.png
Time taken for question: 39 seconds
Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Time taken for question: 21 seconds
Question: Why not use the complete graph for training instead of pruning it?
Tree PNG saved -> decompositions\test6\decomposition_Why_not_use_the_comp_b1b7fff0-69ce-425c-a428-b1aedd43d1d4.png
Time taken for question: 56 seconds
Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for t

In [20]:
max(qa_latencies), min(qa_latencies), sum(qa_latencies)/len(qa_latencies)

(68, 9, 42.36842105263158)

In [19]:
qa_latencies

[9, 39, 21, 56, 47, 62, 47, 39, 57, 32, 58, 43, 35, 45, 68, 43, 62, 25, 17]

In [21]:
all_answers_df = pd.DataFrame(all_answers)
all_answers_df.to_csv("all_answers_hierarchy_final.csv", index=False)

## Evaluation

In [22]:
def f1_token_overlap(pred, truth):
    pred_tokens = set(pred.lower().split())
    truth_tokens = set(truth.lower().split())
    if not pred_tokens or not truth_tokens:
        return 0
    return 2 * len(pred_tokens & truth_tokens) / (len(pred_tokens) + len(truth_tokens))

all_answers_df['f1'] = all_answers_df.apply(
    lambda row: f1_token_overlap(row['answer'], row['ground_truth']), axis=1
)

In [23]:
all_answers_df.sample(2)

,question,answer,context,ground_truth,evidence,source,f1
17,What is the source of the dataset?,Could you specify which dataset you’re asking ...,"{[1] Jia Deng, Wei Dong, Richard Socher, Li-Ji...",The dataset was collected on Youtube.,['Prior research [ 93 ] shows that Youtube vid...,3,0.023256
10,What specific issue with the dot-product atten...,- Issue at large dk: Noam Shazeer and co-autho...,{\[\n\boldsymbol {q} _ {m} ^ {\intercal} \bold...,Noam Shazeer and his co-authors suspected that...,"[""While for small values of dkthe two mechanis...",2,0.363636


In [24]:
df_with_eval = all_answers_df.copy()

In [34]:
from rag_eval import evaluate

results = []
for row in all_answers_df.itertuples():
    print(f"Evaluating Question: {row.question}")
    result = evaluate(
        question=row.question,
        answer=row.answer,
        contexts= row.context,
        ground_truth=row.ground_truth
    )
    results.append(result)

Evaluating Question: What is the target KB size used for MedMentions in the experiments?
Evaluating Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Evaluating Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Evaluating Question: Why not use the complete graph for training instead of pruning it?
Evaluating Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Evaluating Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but mixed results compared to mention-entity baselines on the ZeShEL dataset, and what specific statistical difference accounts for this?
Evaluating Question: During the constraine

In [35]:
for i, result in enumerate(results):
    for metric_name, metric_value in result.items():
        if isinstance(metric_value, dict):
            df_with_eval.at[i, metric_name] = metric_value.get("score")
        else:
            df_with_eval.at[i, metric_name] = metric_value

In [27]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.87500
context_relevance     0.82500
answer_relevance      1.00000
answer_correctness    0.87500
overall               0.89375
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.833333
answer_relevance      0.933333
answer_correctness    0.833333
overall               0.879167
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.000000
context_relevance     0.550000
answer_relevance      1.000000
answer_correctness    0.437500
overall               0.746875
dtype: float64


In [30]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.7500
context_relevance     0.8250
answer_relevance      1.0000
answer_correctness    0.8750
overall               0.8625
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.866667
answer_relevance      0.966667
answer_correctness    0.833333
overall               0.895833
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.0000
context_relevance     0.5500
answer_relevance      1.0000
answer_correctness    0.5000
overall               0.7625
dtype: float64


In [36]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.875000
context_relevance     0.812500
answer_relevance      1.000000
answer_correctness    0.875000
overall               0.890625
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.866667
answer_relevance      0.933333
answer_correctness    0.833333
overall               0.887500
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.000000
context_relevance     0.550000
answer_relevance      1.000000
answer_correctness    0.437500
overall               0.746875
dtype: float64


In [37]:
df_with_eval.to_csv("all_answers_hierarchy_final_with_eval.csv", index=False)